# Prediksi Anomali Suhu ASEAN Berbasis Time Series dan Deep Learning

Notebook ini disusun untuk tugas akhir mata kuliah Data Mining. Fokus utama analisis adalah prediksi perubahan anomali suhu tahunan di kawasan ASEAN menggunakan pendekatan **time series data mining** dan **deep learning**. Model utama yang diuji adalah LSTM dan GRU, sedangkan baseline dan model tabular digunakan sebagai pembanding agar performa deep learning dapat dinilai secara adil.

Data berasal dari FAOSTAT: Temperature Change, Land Cover, dan Emissions Totals. Ruang lingkup notebook dibuat sengaja fokus agar analisis lebih mudah dipertanggungjawabkan.


## 0. Persiapan Environment

Semua output ditulis ke folder `outputs/` dengan subfolder `figures/`, `tables/`, dan `reports_data`. Notebook ini memakai `src/asean_comprehensive_datamining.py` sebagai execution engine supaya pipeline tetap reproducible. Walaupun nama file masih `comprehensive`, pipeline utama sekarang difokuskan pada time series, deep learning, model pembanding, dan forecasting skenario.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import asean_comprehensive_datamining as dm

OUTPUT_DIR = dm.OUTPUT_DIR
dm.ensure_output_dirs(OUTPUT_DIR)
config = dm.ModelingConfig()

print("Output directory:", OUTPUT_DIR.resolve())


## 1. Menjalankan Pipeline Fokus

Cell berikut menjalankan pipeline: data loading, EDA, preprocessing, time series mining, deep learning tuning, perbandingan model, scenario forecasting, visualisasi, dan summary JSON. Jika output sudah ada dan hanya ingin membaca hasil, ubah `RUN_FULL_PIPELINE` menjadi `False`.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    summary = dm.run_pipeline(config)
else:
    summary_path = dm.artifact_path(OUTPUT_DIR, "pipeline_summary.json")
    if not summary_path.exists():
        raise FileNotFoundError("outputs/reports_data/pipeline_summary.json belum ada. Set RUN_FULL_PIPELINE = True lalu jalankan ulang cell ini.")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))

summary


## 2. Pemuatan dan Validasi Data

Panel dibangun dari tiga sumber FAOSTAT: temperature change, land cover, dan emissions. Validasi penting di bagian ini adalah memastikan target delta benar dan tidak ada kolom target masa depan yang masuk sebagai fitur.


In [ ]:
panel = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "faostat_country_year_panel.csv"))
supervised = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "supervised_country_year_modeling.csv"))

print("Panel shape:", panel.shape)
print("Supervised shape:", supervised.shape)
print("Year range:", int(panel["Year"].min()), "-", int(panel["Year"].max()))
print("ASEAN supervised rows:", int(supervised["is_asean"].sum()))

max_delta_error = (
    supervised["target_level_next_year"]
    - supervised["Temperature_Change"]
    - supervised["target_next_year"]
).abs().max()
assert max_delta_error < 1e-10, "Delta target validation failed"
assert supervised.loc[supervised["split"].eq("train"), "Year"].max() <= config.train_end_year
assert supervised.loc[supervised["split"].eq("validation"), "Year"].between(config.train_end_year + 1, config.validation_end_year).all()
assert supervised.loc[supervised["split"].eq("test"), "Year"].min() > config.validation_end_year

print("Delta target and temporal split validation passed.")
supervised[["Area", "Year", "Temperature_Change", "target_level_next_year", "target_next_year", "split", "is_asean"]].head()


## 3. Exploratory Data Analysis

EDA mencakup missing value, uji stasioneritas ADF, korelasi fitur utama ASEAN, tren anomali suhu, dan statistik deskriptif. Bagian ini dipakai untuk memahami kualitas data dan karakter temporal sebelum model LSTM/GRU dibangun.


In [ ]:
missing = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "missing_value_summary.csv"))
adf = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "adf_stationarity_results.csv"))
high_corr = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "high_correlation_pairs_asean.csv"))

print("Top missing columns")
display(missing.head(10))

print("ADF stationarity results")
display(adf)

print("High-correlation feature pairs in ASEAN subset")
display(high_corr.head(12))


In [ ]:
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "correlation_heatmap_asean.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "asean_temperature_trend.png"))))


## 4. Praproses Data

Preprocessing yang diterapkan: missing value handling tiga tahap, log-transform untuk fitur sangat skewed, delta dan percentage-change features, diskretisasi risiko suhu, normalisasi dalam pipeline model, dan seleksi fitur korelasi tinggi berbasis train split. Interpolasi per negara diperlakukan sebagai rekonstruksi data panel dan dicatat sebagai potensi sumber asumsi.


In [ ]:
print("Temperature category distribution in processed supervised data")
display(supervised["Temp_Category"].value_counts(dropna=False).rename_axis("Temp_Category").reset_index(name="count"))

engineered_cols = [c for c in supervised.columns if c.startswith(("delta_", "pct_change_", "log1p_", "cat_"))]
print("Engineered/preprocessing columns:", len(engineered_cols))
print(engineered_cols[:20])


## 5. Deep Learning Baseline

Deep learning adalah fokus utama proyek ini. Model recurrent yang diuji adalah GRU dan LSTM. Keduanya memprediksi target delta suhu tahun berikutnya, lalu prediksi direkonstruksi menjadi level anomali suhu agar dapat dibandingkan dengan nilai aktual.


In [ ]:
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "training_loss_gru.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "training_loss_lstm.png"))))


## 6. Deep Learning Hyperparameter Tuning

Bagian ini membandingkan 288 konfigurasi LSTM/GRU. Konfigurasi dipilih berdasarkan ASEAN validation reconstructed-level MAE, dengan global validation MAE sebagai tie-breaker. Search mencakup sequence window, hidden dimension, learning rate, dropout, recurrent layer depth, weight decay, dan random seed. Test set tetap disimpan untuk pelaporan akhir yang jujur.


In [ ]:
tuning = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "deep_learning_tuning_results.csv"))
tuning_summary = json.loads((dm.artifact_path(OUTPUT_DIR, "deep_learning_tuning_summary.json")).read_text(encoding="utf-8"))

print("Number of tuning experiments:", len(tuning))
print("Selection rule:", tuning_summary["selection_rule"])
print("Best overall DL config:")
display(pd.DataFrame([tuning_summary["best_overall"]]))

assert len(tuning) == 288, "Expected 288 deep learning tuning experiments"
assert tuning["is_default_config"].any(), "Default GRU/LSTM config is missing from tuning results"

tuning[[
    "model_type", "sequence_window", "hidden_dim", "learning_rate",
    "dropout", "num_layers", "weight_decay",
    "epochs_ran", "best_epoch", "final_train_loss", "best_val_loss",
    "asean_validation_reconstructed_level_MAE",
    "validation_reconstructed_level_MAE",
    "asean_test_reconstructed_level_MAE",
    "is_default_config",
]].head(12)


In [ ]:
top10_tuning = tuning.sort_values([
    "asean_validation_reconstructed_level_MAE",
    "validation_reconstructed_level_MAE",
]).head(10)

default_configs = tuning[tuning["is_default_config"]].sort_values([
    "model_type", "asean_validation_reconstructed_level_MAE"
])

print("Top 10 tuned recurrent configs by ASEAN validation MAE")
display(top10_tuning[[
    "model_type", "sequence_window", "hidden_dim", "learning_rate",
    "dropout", "num_layers", "weight_decay",
    "asean_validation_reconstructed_level_MAE",
    "validation_reconstructed_level_MAE",
    "asean_test_reconstructed_level_MAE",
    "is_default_config",
]])

print("Old/default recurrent configs included in the search")
display(default_configs[[
    "model_type", "sequence_window", "hidden_dim", "learning_rate",
    "dropout", "num_layers", "weight_decay",
    "asean_validation_reconstructed_level_MAE",
    "asean_test_reconstructed_level_MAE",
]])


In [ ]:
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "deep_learning_tuning_comparison.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "deep_learning_best_config_loss_gru.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "deep_learning_best_config_loss_lstm.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "deep_learning_tuning_by_parameter.png"))))


### Interpretasi Tuning

Hasil tuning dibaca sebagai bukti konfigurasi recurrent terbaik di antara varian LSTM/GRU yang diuji, bukan sebagai bukti bahwa deep learning selalu lebih unggul. Karena validation dan test ASEAN kecil, hasil tuning berguna untuk rigor metodologis, sedangkan klaim performa akhir tetap memakai held-out test metrics.


In [ ]:
best_dl = tuning.sort_values([
    "asean_validation_reconstructed_level_MAE",
    "validation_reconstructed_level_MAE",
]).iloc[0]

best_by_type = (
    tuning.sort_values([
        "model_type",
        "asean_validation_reconstructed_level_MAE",
        "validation_reconstructed_level_MAE",
    ])
    .groupby("model_type")
    .head(1)
)

print("Best DL model type:", best_dl["model_type"])
print("Best sequence window:", int(best_dl["sequence_window"]))
print("Best hidden dimension:", int(best_dl["hidden_dim"]))
print("Best learning rate:", float(best_dl["learning_rate"]))
print("Best dropout:", float(best_dl["dropout"]))
print("Best recurrent layers:", int(best_dl["num_layers"]))
print("Best weight decay:", float(best_dl["weight_decay"]))
print("ASEAN validation MAE:", round(float(best_dl["asean_validation_reconstructed_level_MAE"]), 4))
print("ASEAN test MAE:", round(float(best_dl["asean_test_reconstructed_level_MAE"]), 4))

print("Best config per recurrent model type")
display(best_by_type[[
    "model_type", "sequence_window", "hidden_dim", "learning_rate",
    "dropout", "num_layers", "weight_decay",
    "asean_validation_reconstructed_level_MAE",
    "asean_test_reconstructed_level_MAE",
]])


## 7. Time Series Mining

Analisis time series mencakup ADF, dekomposisi trend/cycle, ACF/PACF, split temporal, dan supervised forecasting. Karena data bersifat tahunan, dekomposisi tidak diinterpretasikan sebagai seasonal pattern bulanan, melainkan sebagai pemisahan tren dan siklus multi-tahun.


In [ ]:
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "timeseries_decomposition.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "acf_pacf.png"))))


## 8. Model Comparison

Model dievaluasi dengan dua cara: metrik delta untuk target langsung, dan metrik reconstructed level untuk interpretasi suhu aktual tahun berikutnya. Fokus utama adalah LSTM dan GRU, sedangkan baseline, Ridge, Random Forest, dan HistGradientBoosting digunakan sebagai pembanding performa. Ranking utama memakai MAE reconstructed level pada ASEAN test set.


In [ ]:
metrics = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "model_metrics.csv"))
ranking = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "model_ranking_asean_test.csv"))

print("ASEAN test ranking - reconstructed level")
display(ranking)

print("ASEAN test metrics - delta")
display(metrics[(metrics["split"] == "asean_test") & (metrics["target_type"] == "delta")].sort_values("MAE"))

print("ASEAN test metrics - reconstructed level")
display(metrics[(metrics["split"] == "asean_test") & (metrics["target_type"] == "reconstructed_level")].sort_values("MAE"))


In [ ]:
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "model_mae_comparison.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "asean_actual_vs_predicted.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "residual_distribution.png"))))
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "feature_importance_chart.png"))))


## 9. Forecasting Skenario 2023-2030

Scenario forecast memakai model terbaik yang layak untuk roll-forward tabular. Adjustment skenario diperlakukan sebagai asumsi eksplisit atau stress test, bukan proyeksi climate physics yang terkalibrasi. Hasilnya digunakan untuk membaca kemungkinan kategori risiko suhu pada asumsi baseline, emisi tinggi, dan mitigasi-konservasi.


In [ ]:
scenario = pd.read_csv(dm.artifact_path(OUTPUT_DIR, "scenario_2030_predictions.csv"))
display(scenario.head())

display(
    scenario.groupby(["scenario", "predicted_year"])["predicted_temperature_change"]
    .mean()
    .reset_index()
    .tail(12)
)
display(Image(filename=str(dm.artifact_path(OUTPUT_DIR, "asean_scenario_forecast.png"))))


## 10. Final Findings, Unexpected Results, Bias, And Limitations

Bagian ini adalah catatan awal untuk laporan. Angka final harus selalu diambil dari output notebook, bukan diasumsikan sebelum pipeline dijalankan. Fokus pembahasan adalah apakah LSTM/GRU layak untuk data tahunan ASEAN, bagaimana performanya dibanding model pembanding, dan apa keterbatasan forecasting skenario.


In [ ]:
best = ranking.iloc[0]
deep = ranking[ranking["model"].isin(["GRU", "LSTM"])]
adf_nonstationary = int((adf["Kesimpulan"] == "Non-Stasioner").sum())
best_dl = tuning.sort_values([
    "asean_validation_reconstructed_level_MAE",
    "validation_reconstructed_level_MAE",
]).iloc[0]

print("Key findings for focused report")
print("- Best ASEAN reconstructed-level model:", best["model"], "MAE=", round(float(best["MAE"]), 4), "RMSE=", round(float(best["RMSE"]), 4), "n=", int(best["n"]))
print("- Best tuned deep learning config:", best_dl["model_type"], "window=", int(best_dl["sequence_window"]), "hidden=", int(best_dl["hidden_dim"]), "lr=", float(best_dl["learning_rate"]), "dropout=", float(best_dl["dropout"]), "layers=", int(best_dl["num_layers"]), "wd=", float(best_dl["weight_decay"]))
print("- Best tuned DL ASEAN validation MAE:", round(float(best_dl["asean_validation_reconstructed_level_MAE"]), 4), "ASEAN test MAE:", round(float(best_dl["asean_test_reconstructed_level_MAE"]), 4))
print("- ADF result:", adf_nonstationary, "of", len(adf), "ASEAN countries are non-stationary by p >= 0.05.")
if not deep.empty:
    print("- Final deep learning model comparison after tuning:")
    display(deep[["model", "MAE", "RMSE", "R2"]])
print("- Important result: the best deep learning validation configuration is not necessarily the best ASEAN test model, so model comparison with simpler baselines remains necessary.")
print("- Bias/limitation: training on global countries increases data size but may bias fitted patterns away from ASEAN-specific climate-land-emission dynamics.")
print("- Bias/limitation: ASEAN test set has only", int(best["n"]), "samples in the main ranking, so conclusions should be reported as indicative.")
print("- Bias/limitation: Singapore is not available for supervised target evaluation because Temperature_Change is missing in the source data.")
print("- Scenario limitation: 2030 scenarios are illustrative stress tests, not calibrated physical climate projections.")
print("- Method limitation: annual data is not suitable for strong seasonal claims; decomposition is interpreted as trend/cycle analysis.")


## 11. Required Output Check

Cell ini memastikan artefak utama untuk laporan fokus sudah tersedia di `outputs/`. Hanya output yang mendukung deep learning, time series, model comparison, dan skenario yang diwajibkan.


In [ ]:
required_files = [
    "faostat_country_year_panel.csv",
    "supervised_country_year_modeling.csv",
    "model_metrics.csv",
    "model_ranking_asean_test.csv",
    "feature_importance.csv",
    "scenario_2030_predictions.csv",
    "pipeline_summary.json",
    "deep_learning_tuning_results.csv",
    "deep_learning_tuning_summary.json",
    "deep_learning_tuning_by_parameter.csv",
    "correlation_heatmap_asean.png",
    "asean_temperature_trend.png",
    "timeseries_decomposition.png",
    "acf_pacf.png",
    "training_loss_gru.png",
    "training_loss_lstm.png",
    "deep_learning_tuning_comparison.png",
    "deep_learning_tuning_by_parameter.png",
    "model_mae_comparison.png",
    "asean_actual_vs_predicted.png",
    "residual_distribution.png",
    "feature_importance_chart.png",
    "asean_scenario_forecast.png",
]
missing_files = [name for name in required_files if not dm.artifact_path(OUTPUT_DIR, name).exists()]
assert not missing_files, missing_files
print("All required focused-report outputs exist.")
